In [ ]:
import torch
from jet_tagging.datamodules.jetnet_datamodule import JetDataModule
from jet_tagging.models import EncDec
from jet_tagging.modules.joint_ae_module import JointAutoencoderModule
from jet_tagging.utils.visualization import plot_roc_curve, plot_latent_space_tsne

checkpoint_path = "../checkpoints/epoch=14-step=21570-v1.ckpt"  # Update with saved path
model = EncDec(input_size=90, latent_dim=16, output_size=2)
system = JointAutoencoderModule.load_from_checkpoint(checkpoint_path, model=model)
system.eval()

dm = JetDataModule(data_mode="particle", data_dir="../data", batch_size=256)
dm.setup(stage="test")
test_loader = dm.test_dataloader()

all_probs, all_targets, all_latents = [], [], []

with torch.no_grad():
    for x, y in test_loader:
        reconstructed, logits = system(x)
        probs = torch.softmax(logits, dim=-1)[:, 1]
        _, latent = system.model.encoder(x), system.model.encoder(x)  # Extract latent features
        
        all_probs.append(probs)
        all_targets.append(y)
        all_latents.append(latent)

probs_np = torch.cat(all_probs).numpy()
targets_np = torch.cat(all_targets).numpy()
latents_np = torch.cat(all_latents).numpy()

fig_roc = plot_roc_curve(targets_np, probs_np)
fig_tsne = plot_latent_space_tsne(latents_np[:1000], targets_np[:1000])